# Beta Agents Sessions & Session Files — `@azure/ai-projects`

Demonstrates session CRUD and session file operations using the beta agents API.

It mirrors the [`betaAgents.ts`](./betaAgents.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_AGENT_CONTAINER_IMAGE`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  HostedAgentDefinition,
  ProtocolVersionRecord,
  VersionRefIndicator,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const image = process.env["FOUNDRY_AGENT_CONTAINER_IMAGE"] ?? "<agent image>";
console.log(`Agent container image: ${image}`);

Agent container image: "crjep6bl5hlacma.azurecr.io/crjep6bl5hlacma/responses-echo-agent:latest" #crjep6bl5hlacma.azurecr.io/crjep6bl5hlacma/workflow-agent:latest


In [3]:
// Create the AI Project client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

const agentName = "MyBetaAgentsSampleWithHostedAgent13";

In [4]:
// ── Create a hosted agent version ────────────────────────────
console.log("Creating agent...");
const agent = await project.agents.createVersion(
  agentName,
  {
    kind: "hosted",
    cpu: "0.5",
    memory: "1Gi",
    container_configuration: { image: image },
    protocol_versions: [{ protocol: "responses", version: "v1" } as ProtocolVersionRecord],
  } as HostedAgentDefinition,
  {
    metadata: { enableVnextExperience: "true" },
  },
);
console.log(`Agent created (name: ${agent.name}, version: ${agent.version})`);

Creating agent...
Agent created (name: MyBetaAgentsSampleWithHostedAgent13, version: 1)


In [5]:
// Poll until agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, agent.version);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1})`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: active (attempt 1)


In [7]:
// ── Session CRUD ─────────────────────────────────────

// Create a session
const versionIndicator: VersionRefIndicator = {
  type: "version_ref",
  agent_version: agent.version,
};
const session = await project.agents.createSession(agentName, versionIndicator);
console.log(`Session created (id: ${session.agent_session_id}, status: ${session.status})`);

Session created (id: cf4237d0e6f15a5100I5KR0PQMyw2KlnjvAmTpanOPAK2e58gO, status: active)


In [8]:
// Retrieve the session
const fetched = await project.agents.getSession(agentName, session.agent_session_id);
console.log(`Retrieved session (id: ${fetched.agent_session_id}, status: ${fetched.status})`);

Retrieved session (id: cf4237d0e6f15a5100I5KR0PQMyw2KlnjvAmTpanOPAK2e58gO, status: active)


In [9]:
// List sessions
const sessions = [];
const sessionsIterator = project.agents.listSessions(agentName, { limit: 10 })[Symbol.asyncIterator]();
let sessionsNext = await sessionsIterator.next();
while (!sessionsNext.done) {
  const item = sessionsNext.value;
  sessions.push(item);
  sessionsNext = await sessionsIterator.next();
}
console.log(`Found ${sessions.length} session(s)`);
for (const item of sessions) {
  console.log(`  - ${item.agent_session_id} (status: ${item.status})`);
}

Found 4 session(s)
  - 79e64bb6bba3f84300z0pQA2Tn0z6k2t4ufzg4cEFb9ggOVDjH (status: idle)
  - c6cbf7f45100a3170027Be9AA4HRsqBx1UcdvgjDrC70d22sfM (status: active)
  - b88d43042d2e01d400ma92JOt3YY2o8BGmzUzNpFWTutGq9gwH (status: active)
  - cf4237d0e6f15a5100I5KR0PQMyw2KlnjvAmTpanOPAK2e58gO (status: active)
  - 79e64bb6bba3f84300z0pQA2Tn0z6k2t4ufzg4cEFb9ggOVDjH (status: idle)
  - c6cbf7f45100a3170027Be9AA4HRsqBx1UcdvgjDrC70d22sfM (status: active)
  - b88d43042d2e01d400ma92JOt3YY2o8BGmzUzNpFWTutGq9gwH (status: active)
  - cf4237d0e6f15a5100I5KR0PQMyw2KlnjvAmTpanOPAK2e58gO (status: active)


In [10]:
// ── Session File operations ────────────────────────────────

// Upload a file to the session sandbox
const filePath = "/sandbox/hello.txt";
const fileContent = new TextEncoder().encode("Hello from the beta agents sample!");
const uploadResult = await project.agents.uploadSessionFile(
  agentName,
  session.agent_session_id,
  filePath,
  fileContent,
);
console.log(`Uploaded file: ${uploadResult.path} (${uploadResult.bytes_written} bytes)`);

Uploaded file: sandbox/hello.txt (34 bytes)


In [11]:
// List files in the session sandbox (with pagination monitoring)
const files = [];
let pageCount = 0;
const pager = project.agents.listSessionFiles(agentName, session.agent_session_id, {
  path: "/sandbox",
});
const pageIterator = pager.byPage()[Symbol.asyncIterator]();
let pageNext = await pageIterator.next();
while (!pageNext.done) {
  const page = pageNext.value;
  pageCount++;
  console.log(`  Page ${pageCount}: ${page.length} entries`);
  files.push(...page);
  pageNext = await pageIterator.next();
}
console.log(`Files in /sandbox (${files.length} total across ${pageCount} page(s)):`);
for (const entry of files) {
  console.log(
    `  - ${entry.name} (${entry.is_directory ? "directory" : "file"})`,
    JSON.stringify(entry),
  );
}

  Page 1: 1 entries
Files in /sandbox (1 total across 1 page(s)):
  - hello.txt (file) {"name":"hello.txt","size":34,"is_directory":false,"modified_time":"1970-01-01T00:00:00.000Z"}
Files in /sandbox (1 total across 1 page(s)):
  - hello.txt (file) {"name":"hello.txt","size":34,"is_directory":false,"modified_time":"1970-01-01T00:00:00.000Z"}


In [13]:
// Download the file back
const downloadResult = await project.agents.downloadSessionFile(
  agentName,
  session.agent_session_id,
  uploadResult.path,
);

console.log(
  `Downloaded file (has content: ${downloadResult.blobBody !== undefined || downloadResult.readableStreamBody !== undefined})`,
);

Downloaded file (has content: true)


In [14]:
// Delete the file
await project.agents.deleteSessionFile(agentName, session.agent_session_id, filePath);
console.log(`Deleted file: ${filePath}`);

Deleted file: /sandbox/hello.txt


In [18]:
// ── Cleanup ────────────────────────────────────────

// Delete the session
await project.agents.deleteSession(agentName, session.agent_session_id);
console.log("Session deleted");

// Delete the agent version
await project.agents.deleteVersion(agentName, agent.version, { force: true });
console.log("Agent deleted");

Session deleted
Agent deleted
